# MiniMax Prompt Caching via Anthropic SDK

Uses Anthropic SDK with MiniMax-compatible endpoint and tool calling for stable structured output.

In [ ]:
%pip install anthropic

In [16]:
import json
import os

import anthropic

API_KEY = os.getenv("ANTHROPIC_API_KEY") or os.getenv("OPENAI_API_KEY") or "<OPENAI_API_KEY_FROM_ENV>"
BASE_URL = os.getenv("ANTHROPIC_BASE_URL") or "https://api.minimax.io/anthropic"

assert API_KEY, "Set ANTHROPIC_API_KEY or OPENAI_API_KEY in your environment"

client = anthropic.Anthropic(api_key=API_KEY, base_url=BASE_URL)

In [17]:
system_prompt = """
You are an investment decision engine for a retail portfolio app.

Rules:
- Use only the provided selected advisors, risk profile, portfolio state, balance, and market snapshots.
- Be concise and practical.
- Return results only through the forced tool.
- Keep reasons and summaries short.
""".strip()

common_prefix = (
    "Use the same output rules on every request. "
    "Interpret the payload as a portfolio decision task. Input JSON: "
)

tools = [
    {
        "name": "emit_start_recommendations",
        "description": "Return starting buy recommendations and advisor summaries.",
        "input_schema": {
            "type": "object",
            "properties": {
                "buy_recommendations": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "asset_id": {"type": "string"},
                            "allocation_percent": {"type": "string"},
                            "verdict": {"type": "string", "enum": ["buy", "hold", "sell"]},
                            "reason": {"type": "string"}
                        },
                        "required": ["asset_id", "allocation_percent", "verdict", "reason"],
                        "additionalProperties": False
                    }
                },
                "advisor_summaries": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "advisor_id": {"type": "string"},
                            "summary": {"type": "string"}
                        },
                        "required": ["advisor_id", "summary"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["buy_recommendations", "advisor_summaries"],
            "additionalProperties": False
        }
    },
    {
        "name": "emit_portfolio_actions",
        "description": "Return ticker to action mapping for current assets and opportunities.",
        "input_schema": {
            "type": "object",
            "properties": {
                "actions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "asset_id": {"type": "string"},
                            "action": {"type": "string", "enum": ["buy", "hold", "sell", "buy_more"]},
                            "reason": {"type": "string"}
                        },
                        "required": ["asset_id", "action", "reason"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["actions"],
            "additionalProperties": False
        }
    }
]

def print_usage(label, response):
    print(label)
    print(f"Input Tokens: {response.usage.input_tokens}")
    print(f"Output Tokens: {response.usage.output_tokens}")
    print(f"Cache Hit Tokens: {getattr(response.usage, 'cache_read_input_tokens', None)}")

def first_tool_input(response):
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    raise ValueError("No tool_use block in response")

In [24]:
start_payload = {
    "request_type": "start",
    "selected_advisors": [
        {"id": "warren_buffett", "name": "Warren Buffett", "role": "Long-term value investor"},
        {"id": "pavel_durov", "name": "Pavel Durov", "role": "Tech growth and product focus"},
        {"id": "tyler_durden", "name": "Tyler Durden", "role": "High-conviction contrarian"}
    ],
    "risk_profile": "medium",
    "cash_balance_usdt": "1000.00",
    "portfolio": {
        "total_balance_usdt": "1000.00",
        "pnl_percent": "0.00",
        "pnl_absolute": "0.00",
        "assets": []
    },
    "market_snapshots": [
        {"asset_id": "HOODx", "price": "75.13", "upside_percent": "12.40"},
        {"asset_id": "GOOGLx", "price": "314.20", "upside_percent": "8.10"},
        {"asset_id": "NVDAx", "price": "921.50", "upside_percent": "6.25"},
        {"asset_id": "AAPLx", "price": "218.20", "upside_percent": "7.30"}
    ]
}

response1 = client.messages.create(
    model="MiniMax-M2.7-highspeed",
    system=system_prompt,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": common_prefix + json.dumps(start_payload, ensure_ascii=True)
                }
            ]
        }
    ],
    tools=tools,
    tool_choice={"type": "tool", "name": "emit_start_recommendations"},
    max_tokens=1024,
)

print_usage("First request:", response1)
start_result = first_tool_input(response1)
print(json.dumps(start_result, indent=2, ensure_ascii=False))

First request:
Input Tokens: 821
Output Tokens: 606
Cache Hit Tokens: None
{
  "advisor_summaries": [
    {
      "advisor_id": "warren_buffett",
      "summary": "Value-focused strategy prioritizing GOOGLx for long-term fundamentals."
    },
    {
      "advisor_id": "pavel_durov",
      "summary": "Tech growth emphasis; favors HOODx (fintech growth) and GOOGLx."
    },
    {
      "advisor_id": "tyler_durden",
      "summary": "Contrarian high-conviction bet on HOODx with maximum upside potential."
    }
  ],
  "buy_recommendations": [
    {
      "allocation_percent": "40%",
      "asset_id": "GOOGLx",
      "reason": "Strong value play with 8.10% upside; aligns with Buffett's long-term value approach and Durov's tech preference.",
      "verdict": "buy"
    },
    {
      "allocation_percent": "35%",
      "asset_id": "HOODx",
      "reason": "Highest upside at 12.40%; fits Durden's contrarian thesis and Durov's growth focus.",
      "verdict": "buy"
    },
    {
      "allocation_

In [26]:
recommendations_payload = {
    "request_type": "recommendations",
    "selected_advisors": [
        {"id": "warren_buffett", "name": "Warren Buffett", "role": "Long-term value investor"},
        {"id": "pavel_durov", "name": "Pavel Durov", "role": "Tech growth and product focus"},
        {"id": "tyler_durden", "name": "Tyler Durden", "role": "High-conviction contrarian"}
    ],
    "risk_profile": "medium",
    "cash_balance_usdt": "250.00",
    "portfolio": {
        "total_balance_usdt": "1250.00",
        "pnl_percent": "5.20",
        "pnl_absolute": "62.00",
        "assets": [
            {"asset_id": "GOOGLx", "quantity": "1.50", "value_usdt": "471.30", "allocation_percent": "37.70"},
            {"asset_id": "NVDAx", "quantity": "0.50", "value_usdt": "460.75", "allocation_percent": "36.86"}
        ]
    },
    "market_snapshots": [
        {"asset_id": "HOODx", "price": "75.13", "upside_percent": "12.40"},
        {"asset_id": "GOOGLx", "price": "314.20", "upside_percent": "8.10"},
        {"asset_id": "NVDAx", "price": "921.50", "upside_percent": "6.25"},
        {"asset_id": "AAPLx", "price": "218.20", "upside_percent": "7.30"}
    ]
}

response2 = client.messages.create(
    model="MiniMax-M2.7-highspeed",
    system=system_prompt,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": common_prefix + json.dumps(recommendations_payload, ensure_ascii=True)
                }
            ]
        }
    ],
    tools=tools,
    tool_choice={"type": "tool", "name": "emit_portfolio_actions"},
    max_tokens=1024,
)

print_usage("Second request:", response2)
recommendations_result = first_tool_input(response2)
print(json.dumps(recommendations_result, indent=2, ensure_ascii=False))

Second request:
Input Tokens: 892
Output Tokens: 525
Cache Hit Tokens: None
{
  "actions": [
    {
      "action": "buy",
      "asset_id": "HOODx",
      "reason": "Highest upside at 12.4%; adds diversification beyond current tech-heavy portfolio"
    },
    {
      "action": "hold",
      "asset_id": "GOOGLx",
      "reason": "Core holding with 8.1% upside; consider trimming if concentration exceeds comfort"
    },
    {
      "action": "hold",
      "asset_id": "NVDAx",
      "reason": "Strong position with 6.25% upside; maintain for now"
    },
    {
      "action": "buy",
      "asset_id": "AAPLx",
      "reason": "Solid 7.3% upside; adds stable tech exposure and diversification"
    }
  ]
}
